In [1]:
#!pip uninstall tft_pytorch -y

In [2]:
#!pip install -U --no-cache git+https://github.com/rsscml/tft_pytorch

In [3]:
#!pip install -U openpyxl

In [4]:
# base imports
import pandas as pd
import numpy as np
from datetime import datetime, timedelta
import matplotlib.pyplot as plt
import seaborn as sns
import gc
import time
import copy
import os
import random
import warnings
from joblib import Parallel, delayed

# ml scripts
import ml_scripts.config as cfg
from ml_scripts.data_loader import load_and_preprocess

# import TFT
import torch
from tft_pytorch import (
    OptimizedTFTDataset, 
    create_tft_dataloader, 
    create_uniform_embedding_dims,
    TemporalFusionTransformer,
    TFTTrainer,
    TFTInferenceWithTracking)

%matplotlib inline
pd.set_option("display.max_columns", 1000)
pd.set_option("display.max_rows", 1000)
warnings.filterwarnings("ignore")

from disaggregate_forecast_general import DisaggConfig, disaggregate_forecast

In [5]:
DISAGG_DATA_PATH = "../data/plant_mat_channel_agg_input.xlsx"
AGG_DATA_PATH = "../data/plant_matgrp_agg_input.xlsx"

In [6]:
base_df = pd.read_excel(DISAGG_DATA_PATH, dtype={"Plant": 'str', "Material": 'str', "Channel": 'str'})

df = pd.read_excel(AGG_DATA_PATH, dtype={"Plant": 'str', "Channel": 'str'})

# additional feats
df['Month'] = df['YearMonth'].dt.month
df['Quarter'] = df['YearMonth'].dt.quarter

# power transform
df['Customer Demand Qty'] = np.sqrt(df['Customer Demand Qty'])

print(df.groupby(['Plant'])['key'].nunique())


Plant
3891    42
3916     8
Name: key, dtype: int64


In [7]:
df.head()

,Plant,Material_Group,Channel,YearMonth,Customer Demand Qty,Customer Demand Value,Working_Days,Material_Group_Desc,key,Month,Quarter
0,3891,PG011,10,2020-01-01,255.068618,8492467.20,16,Compressors,3891_PG011_10,1,1
1,3891,PG011,10,2020-02-01,222.934968,6390045.92,20,Compressors,3891_PG011_10,2,1
2,3891,PG011,10,2020-03-01,267.822703,9507593.91,22,Compressors,3891_PG011_10,3,1
3,3891,PG011,10,2020-04-01,276.965702,9757240.70,21,Compressors,3891_PG011_10,4,2
4,3891,PG011,10,2020-05-01,282.177249,10250335.63,18,Compressors,3891_PG011_10,5,2


In [8]:
# BG Plant

df = df[df['Plant']=='3916']

df.shape

(432, 11)

In [9]:

# features config
features_config = {
    "entity_col": "key",
    "time_index_col": "YearMonth",
    "target_col": "Customer Demand Qty",

    # known in the future
    "temporal_known_numeric_col_list": ["Working_Days"],
    "temporal_known_categorical_col_list": ["Month", "Quarter"],

    # only historical
    "temporal_unknown_numeric_col_list": [],
    "temporal_unknown_categorical_col_list": [],

    # static per entity
    "static_numeric_col_list": [],
    "static_categorical_col_list": ["Channel","Material_Group"]
}

# context window length
historical_steps=24
forecast_steps=3
train_min_historical_steps=12
test_min_historical_steps=12
infer_min_historical_steps=0
test_periods=6

# no. of samples
train_stride=1
test_stride=1

# parallel processing
train_n_jobs = 4
test_n_jobs = 4
infer_n_jobs = 1

# scaler and encoder location (overwritten for each eval run)
scaler_path='./artefacts/train_scalers.pkl'
scaling_strategy='entity_level'
scaling_method='mean'
encoders_path='./artefacts'

# train cutoff points for various eval runs
ORIGINS = [
    (pd.Timestamp("2024-07-01"), pd.Timestamp("2025-01-01"), pd.Timestamp("2025-04-01")), # -> target April-2025
    (pd.Timestamp("2024-08-01"), pd.Timestamp("2025-02-01"), pd.Timestamp("2025-05-01")), # -> target May-2025
    (pd.Timestamp("2024-09-01"), pd.Timestamp("2025-03-01"), pd.Timestamp("2025-06-01")), # -> target Jun-2025
    (pd.Timestamp("2024-10-01"), pd.Timestamp("2025-04-01"), pd.Timestamp("2025-07-01")), # -> target Jul-2025
    (pd.Timestamp("2024-11-01"), pd.Timestamp("2025-05-01"), pd.Timestamp("2025-08-01")), # -> target Aug-2025
    (pd.Timestamp("2024-12-01"), pd.Timestamp("2025-06-01"), pd.Timestamp("2025-09-01")), # -> target Sep-2025
    (pd.Timestamp("2025-01-01"), pd.Timestamp("2025-07-01"), pd.Timestamp("2025-10-01")), # -> target Oct-2025
    (pd.Timestamp("2025-02-01"), pd.Timestamp("2025-08-01"), pd.Timestamp("2025-11-01")), # -> target Nov-2025
    (pd.Timestamp("2025-03-01"), pd.Timestamp("2025-09-01"), pd.Timestamp("2025-12-01")), # -> target Dec-2025
]


In [10]:
# Eval loop

disagg_lag3_eval_df_list = []
lag3_eval_df_list = []

for i, origin in enumerate(ORIGINS):

    print(f"Eval run: {i+1}")
    
    # ----------------- split dataframe ------------------------
    train_start = pd.to_datetime('2020-01-01', format="%Y-%m-%d")
    train_cutoff = origin[0]
    
    test_start = train_cutoff - pd.DateOffset(months=historical_steps)
    test_cutoff = origin[1]

    infer_cutoff = origin[2]
    infer_start = infer_cutoff - pd.DateOffset(months=historical_steps + forecast_steps)

    print(f" train_start: {train_start}, train_cutoff: {train_cutoff}")
    print(f" test_start: {test_start}, test_cutoff: {test_cutoff}")
    print(f" infer_start: {infer_start}, infer_cutoff: {infer_cutoff}")
    
    train_df = df[df['YearMonth'] <= train_cutoff].copy()
    test_df = df[(df['YearMonth'] > test_start) & (df['YearMonth'] <= test_cutoff)].copy()
    
    infer_df = df[df['YearMonth'] <= infer_cutoff].copy()
    infer_df =  infer_df.groupby(['key'], group_keys=False).tail(historical_steps + forecast_steps)
    
    print(train_df.shape, test_df.shape, infer_df.shape)

    # ------------------ create datasets ----------------------------
    train_dataset = OptimizedTFTDataset(
                        data_source=train_df,
                        features_config=features_config, 
                        historical_steps=historical_steps,
                        prediction_steps=forecast_steps,
                        stride=train_stride,
                        enable_padding=True,
                        padding_strategy = 'zero',
                        categorical_padding_value = -1,
                        min_historical_steps=train_min_historical_steps,
                        allow_inference_only_entities=False,
                        scaler_path=scaler_path,
                        scaling_strategy=scaling_strategy,
                        scaling_method=scaling_method,
                        cold_start_scaler_cols=None,
                        mean_scaler_epsilon=1.0,
                        recency_alpha=1.0,
                        n_jobs=train_n_jobs,
                        max_series=None,
                        mode='train',
                        encoders_path=encoders_path,
                        fit_encoders=None,
                        preprocessing_fn=None)
    
    test_dataset = OptimizedTFTDataset(
                        data_source=test_df,
                        features_config=features_config, 
                        historical_steps=historical_steps,
                        prediction_steps=forecast_steps,
                        stride=test_stride,
                        enable_padding=True,
                        padding_strategy = 'zero',
                        categorical_padding_value = -1,
                        min_historical_steps=test_min_historical_steps,
                        allow_inference_only_entities=False,
                        scaler_path=scaler_path,
                        scaling_strategy=scaling_strategy,
                        scaling_method=scaling_method,
                        cold_start_scaler_cols=None,
                        mean_scaler_epsilon=1.0,
                        recency_alpha=0,
                        n_jobs=test_n_jobs,
                        max_series=None,
                        mode='test',
                        encoders_path=encoders_path,
                        fit_encoders=None,
                        preprocessing_fn=None)

    infer_dataset = OptimizedTFTDataset(
                        data_source=infer_df,
                        features_config=features_config, 
                        historical_steps=historical_steps,
                        prediction_steps=forecast_steps,
                        stride=test_stride,
                        enable_padding=True,
                        padding_strategy = 'zero',
                        categorical_padding_value = -1,
                        min_historical_steps=infer_min_historical_steps,
                        allow_inference_only_entities=True,
                        scaler_path=scaler_path,
                        scaling_strategy=scaling_strategy,
                        scaling_method=scaling_method,
                        cold_start_scaler_cols=['Material_Group','Channel'], 
                        mean_scaler_epsilon=1.0,
                        recency_alpha=0,
                        n_jobs=1,
                        max_series=None,
                        mode='test',
                        encoders_path=encoders_path,
                        fit_encoders=None,
                        preprocessing_fn=None)

    # ------------------------ create dataloaders -------------------------------
    train_dataloader, train_adapter = create_tft_dataloader(train_dataset, batch_size=64, shuffle=True, num_workers=0, drop_last=False, pin_memory=True)

    test_dataloader, test_adapter = create_tft_dataloader(test_dataset, batch_size=64, shuffle=False, num_workers=0, drop_last=False, pin_memory=True)

    infer_dataloader, infer_adapter = create_tft_dataloader(infer_dataset, batch_size=64, shuffle=False, num_workers=0, drop_last=False, pin_memory=True)

    # ------------------- create embedding dims dict for various categorical variables
    categorical_embedding_dims = create_uniform_embedding_dims(train_dataset, hidden_layer_size=160)
    num_static_categorical=len(train_dataset.static_categorical_cols)
    num_static_continuous=len(train_dataset.static_numeric_cols)
    num_historical_categorical=len(train_dataset.temporal_unknown_categorical_cols) + len(train_dataset.temporal_known_categorical_cols)
    num_historical_continuous=len(train_dataset.target_cols) + len(train_dataset.temporal_unknown_numeric_cols) + len(train_dataset.temporal_known_numeric_cols)
    num_future_categorical=len(train_dataset.temporal_known_categorical_cols)
    num_future_continuous=len(train_dataset.temporal_known_numeric_cols)

    # ---------------------- create model -----------------------------------
    model = TemporalFusionTransformer(
            # Model architecture parameters
            hidden_layer_size = 160,
            num_attention_heads = 1,
            num_lstm_layers = 1,
            num_attention_layers = 4,
            dropout_rate = 0.1,
            
            # Input specification
            num_static_categorical = num_static_categorical,
            num_static_continuous = num_static_continuous,
            num_historical_categorical = num_historical_categorical,
            num_historical_continuous = num_historical_continuous,
            num_future_categorical = num_future_categorical,
            num_future_continuous = num_future_continuous,
        
            # embedding dims for categorical variables
            categorical_embedding_dims = categorical_embedding_dims,
            
            # Time dimensions
            historical_steps = historical_steps,
            prediction_steps = forecast_steps,
            
            # Output specification
            num_outputs = 1,
            device = 'cuda' #'cuda' if torch.cuda.is_available() else 'cpu'
        )
    
    # ------------------- create trainer ------------------------------------
    trainer = TFTTrainer(model = model,
                     # data params
                     train_loader = train_dataloader,
                     val_loader = test_dataloader,
                     train_adapter = train_adapter,
                     val_adapter = test_adapter,
                     # loss params
                     loss_type = 'huber',
                     loss_params = {'delta': 1.0},
                     # optimizer params
                     optimizer_type = 'adam',
                     learning_rate = 0.0001,
                     # Scheduler configuration
                     scheduler_type = 'reduce_on_plateau',
                     scheduler_factor = 0.5,
                     scheduler_patience = 5,
                     # Training options
                     enable_gradient_clipping = True,
                     max_grad_norm = 1.0,
                     enable_train_sample_weighting = False,
                     enable_val_sample_weighting = False,
                     # Mixed Precision Training
                     enable_mixed_precision = False,
                     # Checkpointing
                     save_path = './models/bg_plant_matgrp_ch',
                     save_every = 1)
    
    # ------------------------ train -----------------------------------------
    trainer.train(num_epochs=50, patience=10)

    # ----------------------- infer ------------------------------------------
    inferencewithtracking = TFTInferenceWithTracking(model_path = 'models/bg_plant_matgrp_ch/best_model.pt', model = model, adapter = infer_adapter, device = 'cuda')
    results_df = inferencewithtracking.predict_with_metadata(infer_dataloader)

    # ----------------------- inverse power transform -------------------------
    results_df['pred_0'] = np.square(results_df['pred_0'])
    results_df['actual_Customer Demand Qty'] = np.square(results_df['actual_Customer Demand Qty'])
    results_df.rename(columns={'entity_id':'key', 'timestamp': 'YearMonth'}, inplace=True)

    # get lag3
    lag3_results_df = results_df[results_df['YearMonth']==origin[2]]
    
    # merge with base columns
    infer_df = infer_df[['key','Plant','Material_Group','Channel']]
    infer_df.drop_duplicates(inplace=True)
    lag3_results_df = lag3_results_df.merge(infer_df, on=['key'], how='left')
    
    # write out
    lag3_results_df.to_csv(f"./outputs/bg_hub1_transform_plant_matgrp_channel_{str(origin[2])}.csv")
    
    lag3_eval_df_list.append(lag3_results_df)

    # disaggregate to Material
    cfg = DisaggConfig(
        group_key_cols=["Plant", "Material_Group", "Channel"], # key columns in training dataset
        item_col="Material", # disaggregation level (Plant, Material, Channel)
        time_col="YearMonth",
        historical_qty_col="Customer Demand Qty",
        forecast_qty_col="pred_0",
        output_qty_col="Disaggregated Forecast Qty",
        proportion_col="proportion",
        output_key_col="key",
        output_key_parts=["Plant", "Material", "Channel"],
        output_key_sep="_",
    )

    disagg_lag3_results_df = disaggregate_forecast(
        forecast_df=lag3_results_df,
        historical_df=base_df,
        config=cfg,
        cutoff_date=origin[1] + pd.DateOffset(months=1), # first month of forecast
        lookback_months=6,
    )

    disagg_lag3_eval_df_list.append(disagg_lag3_results_df)

    # ----------------------- clear model ------------------------------------- 
    del model
    gc.collect()



print("Combine & save results")
results_df = pd.concat(lag3_eval_df_list, ignore_index=True)
disagg_results_df = pd.concat(disagg_lag3_eval_df_list, ignore_index=True)

# merge actuals at disagg level for comparison
base_df = base_df[['key','YearMonth','Customer Demand Qty']].drop_duplicates()
disagg_results_df = disagg_results_df.merge(base_df, on=['key','YearMonth'], how='left')

# Save
results_df.to_csv("./outputs/bg_hub1_transform_plant_matgrp_channel_lag3eval.csv", index=False)
disagg_results_df.to_csv("./outputs/bg_hub1_disagg_plant_material_channel_lag3eval.csv", index=False)


Eval run: 1
 train_start: 2020-01-01 00:00:00, train_cutoff: 2024-07-01 00:00:00
 test_start: 2022-07-01 00:00:00, test_cutoff: 2025-01-01 00:00:00
 infer_start: 2023-01-01 00:00:00, infer_cutoff: 2025-04-01 00:00:00
(330, 11) (180, 11) (210, 11)

Feature Configuration:
  Targets: 1
  Static: 0 numeric, 2 categorical
  Known temporal: 1 numeric, 2 categorical
  Unknown temporal: 0 numeric, 0 categorical
  Total numeric features: 2
Will save scalers to ./artefacts/train_scalers.pkl after fitting
Loading data...
Processing 8 entities...
  Using sequential processing...
Loaded 8 valid series from 8 total
Computing padding values for each entity...
Computed padding values for 8 entities
Fitting encoders on train data...
  Fitting encoder for Channel...
    -> 3 unique values
  Fitting encoder for Material_Group...
    -> 5 unique values
  Fitting encoder for Month...
    -> 12 unique values
  Fitting encoder for Quarter...
    -> 4 unique values
Encoders saved to artefacts
Creating windows

2026-03-25 23:03:57,901 - INFO - Starting training for 50 epochs...
2026-03-25 23:03:57,901 - INFO - 
Epoch 1/50
2026-03-25 23:03:58,323 - INFO - Batch 0/2, Loss: 0.6375
2026-03-25 23:03:58,362 - INFO - Train Loss: 0.5959
2026-03-25 23:03:58,374 - INFO - Val Loss: 10.9259, Metrics: {'mse': 1260.5284423828125, 'rmse': 35.503921507106966, 'mae': 11.379518508911133, 'mape': nan, 'val_loss': 10.925884246826172}
2026-03-25 23:03:58,375 - INFO - New best validation loss: 10.9259
2026-03-25 23:03:58,479 - INFO - Saved checkpoint to models/bg_plant_matgrp_ch/checkpoint_epoch_1.pt
2026-03-25 23:03:58,571 - INFO - Saved best model to models/bg_plant_matgrp_ch/best_model.pt
2026-03-25 23:03:58,628 - INFO - 
Epoch 2/50
2026-03-25 23:03:58,663 - INFO - Batch 0/2, Loss: 0.4260
2026-03-25 23:03:58,694 - INFO - Train Loss: 0.4522
2026-03-25 23:03:58,704 - INFO - Val Loss: 10.7439, Metrics: {'mse': 1255.4656982421875, 'rmse': 35.432551393347154, 'mae': 11.181710243225098, 'mape': nan, 'val_loss': 10.74

Computed proportions for 5449 item-within-group combinations.

Aggregated forecast total : 1,177,066.75
Disaggregated forecast total: 1,177,066.66
Difference (leakage)       : 0.09


2026-03-25 23:04:07,174 - INFO - Starting training for 50 epochs...
2026-03-25 23:04:07,175 - INFO - 
Epoch 1/50
2026-03-25 23:04:07,254 - INFO - Batch 0/3, Loss: 0.2565


Eval run: 2
 train_start: 2020-01-01 00:00:00, train_cutoff: 2024-08-01 00:00:00
 test_start: 2022-08-01 00:00:00, test_cutoff: 2025-02-01 00:00:00
 infer_start: 2023-02-01 00:00:00, infer_cutoff: 2025-05-01 00:00:00
(336, 11) (180, 11) (210, 11)

Feature Configuration:
  Targets: 1
  Static: 0 numeric, 2 categorical
  Known temporal: 1 numeric, 2 categorical
  Unknown temporal: 0 numeric, 0 categorical
  Total numeric features: 2
Will save scalers to ./artefacts/train_scalers.pkl after fitting
Loading data...
Processing 8 entities...
  Using sequential processing...
Loaded 8 valid series from 8 total
Computing padding values for each entity...
Computed padding values for 8 entities
Fitting encoders on train data...
  Fitting encoder for Channel...
    -> 3 unique values
  Fitting encoder for Material_Group...
    -> 5 unique values
  Fitting encoder for Month...
    -> 12 unique values
  Fitting encoder for Quarter...
    -> 4 unique values
Encoders saved to artefacts
Creating windows

...

Computed proportions for 5449 item-within-group combinations.

Aggregated forecast total : 883,746.06
Disaggregated forecast total: 883,745.68
Difference (leakage)       : 0.38


2026-03-25 23:05:29,313 - INFO - Starting training for 50 epochs...
2026-03-25 23:05:29,314 - INFO - 
Epoch 1/50
2026-03-25 23:05:29,385 - INFO - Batch 0/3, Loss: 7.3658


Eval run: 9
 train_start: 2020-01-01 00:00:00, train_cutoff: 2025-03-01 00:00:00
 test_start: 2023-03-01 00:00:00, test_cutoff: 2025-09-01 00:00:00
 infer_start: 2023-09-01 00:00:00, infer_cutoff: 2025-12-01 00:00:00
(378, 11) (180, 11) (210, 11)

Feature Configuration:
  Targets: 1
  Static: 0 numeric, 2 categorical
  Known temporal: 1 numeric, 2 categorical
  Unknown temporal: 0 numeric, 0 categorical
  Total numeric features: 2
Will save scalers to ./artefacts/train_scalers.pkl after fitting
Loading data...
Processing 8 entities...
  Using sequential processing...
Loaded 8 valid series from 8 total
Computing padding values for each entity...
Computed padding values for 8 entities
Fitting encoders on train data...
  Fitting encoder for Channel...
    -> 3 unique values
  Fitting encoder for Material_Group...
    -> 5 unique values
  Fitting encoder for Month...
    -> 12 unique values
  Fitting encoder for Quarter...
    -> 4 unique values
Encoders saved to artefacts
Creating windows

2026-03-25 23:05:29,504 - INFO - Train Loss: 3.6260
2026-03-25 23:05:29,522 - INFO - Val Loss: 34.5798, Metrics: {'mse': 7191.4150390625, 'rmse': 84.8022112863957, 'mae': 34.91358184814453, 'mape': nan, 'val_loss': 34.579833984375}
2026-03-25 23:05:29,523 - INFO - New best validation loss: 34.5798
2026-03-25 23:05:29,642 - INFO - Saved checkpoint to models/bg_plant_matgrp_ch/checkpoint_epoch_1.pt
2026-03-25 23:05:29,748 - INFO - Saved best model to models/bg_plant_matgrp_ch/best_model.pt
2026-03-25 23:05:29,818 - INFO - 
Epoch 2/50
2026-03-25 23:05:29,856 - INFO - Batch 0/3, Loss: 6.8696
2026-03-25 23:05:29,934 - INFO - Train Loss: 3.5733
2026-03-25 23:05:29,947 - INFO - Val Loss: 34.5113, Metrics: {'mse': 7185.86279296875, 'rmse': 84.76946851885265, 'mae': 34.80963134765625, 'mape': nan, 'val_loss': 34.51127624511719}
2026-03-25 23:05:29,947 - INFO - New best validation loss: 34.5113
2026-03-25 23:05:30,068 - INFO - Saved checkpoint to models/bg_plant_matgrp_ch/checkpoint_epoch_2.pt
2

Computed proportions for 5449 item-within-group combinations.

Aggregated forecast total : 1,111,608.75
Disaggregated forecast total: 1,111,591.41
Difference (leakage)       : 17.34
Combine & save results


In [ ]:
disagg_results_df.head()

In [ ]:
base_df.head()

In [ ]:
disagg_results_df.shape

In [ ]:
base_df = base_df[['key','YearMonth','Customer Demand Qty']].drop_duplicates()

disagg_results_df = disagg_results_df.merge(base_df, on=['key','YearMonth'], how='left')

In [ ]:
disagg_results_df.to_csv("./outputs/ch_hub1_disagg_plant_material_channel_lag3eval_actuals.csv", index=False)